# Real-Time Foreground Segmentation and Selective Blur, on a GPU

**CSC14116 — Applied Parallel Programming** · Huỳnh Lê Hải Dương (22127081), Nguyễn Đức Tín (22127415)

Repository: https://github.com/Sn-cpp/Adaptive-Gaussian-Mixture-Model

---

Separate the moving vehicles from a fixed traffic camera's background, then blur the road and
keep the cars sharp. The segmentation is an adaptive Gaussian Mixture Model (Zivkovic, 2004);
the interesting part of the project is not that it works but **where the time actually goes**,
which turned out not to be where we assumed.

**On the numbers.** Everything this notebook can recompute, it recomputes — the parity tests,
the Q8 derivation, the fixed-point comparison against OpenCV, the flood-fill benchmark, the mask
figures. Three sets of numbers it cannot, and they are labelled where they appear:

| | why it is quoted rather than run |
|---|---|
| **F1 / IoU on CDnet** | the dataset host (`changedetection.net`) no longer resolves. Measured at commit `b2523ba` with the same host chain this checkout ships; re-run with `eval_highway.py` if you have a copy. |
| **T4 timings** | need a GPU. Reproduce with `bench_post.py` and `bench_t4.py` on one; raw output is in `RESULTS-T4.md`. |
| **the 81.8%-on-host profile** | a one-off measurement from earlier in the project, kept because it is what motivated v1. No profiler in this repo reproduces it. |

Nothing else is transcribed.

**Runtime → Change runtime type → T4 GPU** before running. Without a GPU the parity and quality
sections still run; the benchmark section will say so and skip.

## 0. Setup

In [ ]:
import os, subprocess, sys
REPO   = "https://github.com/Sn-cpp/Adaptive-Gaussian-Mixture-Model"
BRANCH = "dev/HD-car"

# Resolve by looking for the repo itself, not by testing a relative path: if this
# notebook is already running inside the clone, `os.path.isdir("project")` is
# False and the naive version clones a second copy *inside* it.
def repo_root():
    for c in (os.getcwd(), "/content/project", os.path.join(os.getcwd(), "project")):
        if os.path.isdir(os.path.join(c, ".git")) and \
           os.path.exists(os.path.join(c, "gmm_mask")):
            return c
    return None

root = repo_root()
if root is None:
    dest = "/content/project" if os.path.isdir("/content") else "project"
    subprocess.run(["git","clone","--branch",BRANCH,"--depth","1",REPO,dest], check=True)
    root = dest
os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)
print("repo:", root)
print(subprocess.run(["git","log","--oneline","-1"],capture_output=True,text=True).stdout.strip())

In [ ]:
# numba<0.62 so `numba.cuda` still resolves against Colab's numba-cuda package.
!pip -q install "numba<0.62" 2>&1 | tail -2

import platform, numpy as np, cv2, numba
print(f"python {platform.python_version()} | numpy {np.__version__} | "
      f"opencv {cv2.__version__} | numba {numba.__version__}")

from numba import cuda
HAS_GPU = cuda.is_available()
if HAS_GPU:
    d = cuda.get_current_device()
    # numba hands back bytes on some versions and str on others
    name = d.name.decode() if isinstance(d.name, bytes) else str(d.name)
    print(f"GPU: {name} (compute {d.compute_capability})")
else:
    print("NO GPU — set Runtime > Change runtime type > T4. Benchmarks will be skipped.")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi":120, "font.size":9, "axes.grid":True,
                     "grid.alpha":0.3, "axes.spines.top":False, "axes.spines.right":False})
C = {"v0":"#c44e52", "v1":"#4c72b0", "v2":"#55a868", "host":"#8172b2"}

## 1. The problem, and why it belongs on a GPU

At 1920×1080 each frame holds 2 073 600 pixels. The model keeps **K=5 Gaussians per pixel**,
each with a weight, a 3-channel mean and a variance — 5 + 15 + 5 = **25 float32 per pixel**,
so 51.8 M values and **207 MB of model state resident, touched every frame**. A 15×15 Gaussian
blur adds 30 multiply-accumulates per background pixel once separated into two passes.

Sequential Python is nowhere near real time: `bench_post.py --with-sequential` measures
**3035 ms/frame at 480p** on a Colab T4 host, 0.3 FPS. Real time needs 30 at 1080p.

The structure is unusually friendly to a GPU: the GMM update has *no cross-pixel dependency at
all*, so a 1080p frame is 2 073 600 independent threads. The mask stages are per-pixel or small
stencils. Exactly one stage resists, and §7 is about that one.

## 2. The dataset and the scoring protocol

**CDnet 2014 `baseline/highway`** — 1700 frames at 320×240 of a fixed traffic camera, with
per-frame hand-labelled ground truth.

The protocol is not a detail. CDnet ships a region of interest (`ROI.bmp`) and a temporal window
(`temporalROI.txt` = frames 470–1700), and it labels **shadows as 50 and object boundaries as
170, both defined as _don't care_**. Scoring those pixels is the easiest way to publish a wrong
number, so they are excluded explicitly. Changing the window alone moves F1 by up to 5 points —
it was the single largest source of contradictory measurements in this project.

In [ ]:
import os, glob, socket, urllib.request, zipfile

# urlretrieve has no default timeout: on a network that blackholes the SYN
# rather than refusing it, this cell hangs forever and the notebook never
# finishes. Bound it.
socket.setdefaulttimeout(20)

# Honour an existing HIGHWAY_DIR before reaching for the network, and accept a
# zip the user uploaded by hand. As of this writing the CDnet host no longer
# resolves, so the manual paths are the ones that actually work.
HIGHWAY = os.environ.get("HIGHWAY_DIR", "highway")

def usable(d):
    return os.path.isdir(os.path.join(d, "input")) and \
           os.path.isdir(os.path.join(d, "groundtruth"))

if not usable(HIGHWAY):
    local = glob.glob("highway.zip") + glob.glob("/content/highway.zip") + \
            glob.glob("/content/drive/MyDrive/highway.zip")
    if local:
        print("extracting", local[0])
        with zipfile.ZipFile(local[0]) as z: z.extractall(".")
    else:
        url = "http://wordpress-jodoin.dmi.usherb.ca/static/dataset/baseline/highway.zip"
        try:
            print("trying", url)
            urllib.request.urlretrieve(url, "highway.zip")
            with zipfile.ZipFile("highway.zip") as z: z.extractall(".")
        except Exception as e:
            print(f"  unavailable: {type(e).__name__}: {e}")

HAS_DATA = usable(HIGHWAY)
os.environ["HIGHWAY_DIR"] = HIGHWAY
print("dataset present:", HAS_DATA, "at", HIGHWAY)
if not HAS_DATA:
    print()
    print("  The CDnet download host is offline. To score quality, put a copy of")
    print("  the `highway` sequence somewhere this notebook can see it -- upload")
    print("  highway.zip to /content, or mount Drive -- and re-run this cell.")
    print("  Sections 2, 4, 5 and 8 will be skipped until then; everything else")
    print("  (parity, the Q8 derivation, the benchmarks) runs without it.")

# The figures below need a sequence of frames; only the *scoring* needs ground
# truth. So the source degrades rather than disappearing: CDnet if present, a
# bundled clip if one is (the .mp4 files are gitignored, so a fresh clone has
# none), and otherwise the same synthetic traffic `bench_post.py` benchmarks
# on. A report whose pictures all vanish because a download host went offline
# is not much of a report.
import cv2, numpy as np

def load_sequence(n, size=(320, 240)):
    """n BGR frames, from the best source available."""
    if HAS_DATA:
        out = [cv2.imread(f"{HIGHWAY}/input/in{i:06d}.jpg") for i in range(1, n + 1)]
        out = [f for f in out if f is not None]
        if out:
            return out, "CDnet highway (ground truth available)"
    for clip in ("LTSSUD-Test.mp4", "TestStableBackground.mp4"):
        if not os.path.exists(clip):
            continue
        cap, out = cv2.VideoCapture(clip), []
        while len(out) < n:
            ok, f = cap.read()
            if not ok:
                break
            out.append(cv2.resize(f, size))
        cap.release()
        if out:
            return out, f"{clip} (no ground truth — figures only)"
    import bench_post
    return (bench_post.make_frames(n, size),
            "synthetic traffic from bench_post.make_frames (no ground truth)")

_probe, SEQ_NAME = load_sequence(2)
HAS_FRAMES = bool(_probe)
print("frame source for the figures:", SEQ_NAME)

In [ ]:
if HAS_DATA:
    import cv2, numpy as np, matplotlib.pyplot as plt
    i = 1200
    frame = cv2.imread(f"{HIGHWAY}/input/in{i:06d}.jpg")
    gt    = cv2.imread(f"{HIGHWAY}/groundtruth/gt{i:06d}.png", 0)
    roi   = cv2.imread(f"{HIGHWAY}/ROI.bmp", 0)

    fig, ax = plt.subplots(1, 3, figsize=(11, 2.8))
    ax[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); ax[0].set_title(f"input frame {i}")
    ax[1].imshow(gt, cmap="gray");  ax[1].set_title("ground truth\n0 bg · 50 shadow · 170 unknown · 255 fg")
    ax[2].imshow(roi, cmap="gray"); ax[2].set_title("ROI — scored region")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

    vals, counts = np.unique(gt, return_counts=True)
    print("ground-truth labels in this frame:",
          {int(v): int(c) for v, c in zip(vals, counts)})
    print("-> the 50s and 170s are excluded from every score below")

## 3. Four implementations, one algorithm

| backend | what it is |
|---|---|
| `GMM_Mask_CPU` | the specification: a per-pixel Python loop, transliterated from Zivkovic |
| `GMM_Mask_Numba` | the same, `@njit` with `prange` |
| `GMM_Mask_CUDA` (v0) | one thread per pixel, planar coalesced state |
| `GMM_Mask_CUDA_v1/v2` | v0 plus the post-processing and blur on the device |

The claim is that they produce **identical masks**. That claim used to have no test behind it —
the only surviving test compared v1 against v2, the narrowest of the six pairs. A benchmark
table across four backends whose outputs were never compared is a table of four different
algorithms, so this runs first.

In [ ]:
!python -m pytest tests/test_parity.py -q 2>&1 | tail -8

### 3.1 Against OpenCV itself

The 75% deliverable is agreement with `cv2.createBackgroundSubtractorMOG2`. Measured, and
reported with the qualification it needs:

* **synthetic sequences: bit-identical** — 0 of 30 720 pixels over 20 frames, 0 of 92 160 under
  heavy noise, 0 of 204 800 at 64×80.
* **real video: 22 pixels of 1 536 000 (0.0014%)**, all 22 in a single frame; 1 pixel of
  1 536 000 on the second clip.

The residue is the float32 boundary. OpenCV accumulates in a different order and contracts its
own FMAs, so a pixel sitting within an ulp of `Tb·σ²` lands on either side of the comparison.
Synthetic frames put almost nothing that close to the threshold; camera noise does.

"Bit-exact on synthetic input, 0.0014% disagreement on video" is true. "Bit-exact" unqualified
is not, and the difference is one frame of one clip.

In [ ]:
import numpy as np, cv2
from gmm_mask import GMM_Mask_Numba
from settings import (MOG2_HISTORY, MOG2_VAR_THRESHOLD, MOG2_N_COMPONENTS,
                      MOG2_BACKGROUND_RATIO)

if HAS_DATA:
    ours = None; cvm = cv2.createBackgroundSubtractorMOG2(
        history=int(MOG2_HISTORY), varThreshold=float(MOG2_VAR_THRESHOLD),
        detectShadows=False)
    cvm.setNMixtures(int(MOG2_N_COMPONENTS)); cvm.setBackgroundRatio(float(MOG2_BACKGROUND_RATIO))
    per_frame = []
    for i in range(1, 121):
        bgr = cv2.imread(f"{HIGHWAY}/input/in{i:06d}.jpg")
        if ours is None: ours = GMM_Mask_Numba(*bgr.shape[:2])
        m_ours, _, _ = ours.apply(np.ascontiguousarray(bgr, np.float32))
        per_frame.append(int((np.asarray(m_ours) != cvm.apply(bgr)).sum()))
    px = 120 * bgr.shape[0] * bgr.shape[1]
    print(f"highway, first 120 frames: {sum(per_frame)} of {px} pixels differ "
          f"({sum(per_frame)/px:.5%}); worst frame {max(per_frame)} px")

    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,2.2))
    plt.plot(per_frame, lw=0.9, color=C["v1"])
    plt.xlabel("frame"); plt.ylabel("pixels differing\nfrom cv2.MOG2")
    plt.title("Disagreement with OpenCV is a boundary effect, not a drift")
    plt.tight_layout(); plt.show()

## 4. Colour space: the single largest quality factor

Measured on `highway` with an identical downstream chain:

| model input | F1 |
|---|---|
| BGR | 0.827 |
| **YCrCb** | **0.984** |

Separating luma from chroma lets a car's shadow — which changes brightness but not colour —
fail the match without dragging the car with it. It costs one `cvtColor` and touches no kernel,
and it is worth more than every post-processing step in this notebook combined.

In [ ]:
if HAS_DATA:
    !python eval_highway.py --colorspace both --last-frame 700 2>&1 | tail -24
    print("\n(--last-frame 700 for notebook runtime; the reported numbers use the full 1700)")

## 5. Post-processing: what we measured and rejected

Raw MOG2 output is speckled and hollow. Sensor noise flips single pixels, and inside a large
uniform surface — a car roof — nothing moves, so the middle comes back empty.

Three things the measurements decided, **all of which had been assumed the other way**.

> *Quoted, not run here* — measured at commit `b2523ba` on the full 470–1700 window. The chain
> in this checkout is unchanged (`utils/post_processing.refine_mask`), but CDnet is no longer
> downloadable, so the cell below cannot re-score them.

| chain | F1 | IoU | P | R | empty |
|---|---|---|---|---|---|
| `bg_prob < 0.5` + median5 + **fill** | **0.9843** | 0.9691 | 0.9863 | 0.9823 | 0 |
| `bg_prob < 0.5` + median5 | 0.9805 | 0.9617 | 0.9863 | 0.9748 | 0 |
| + morphological CLOSE 15 | 0.9782 | 0.9572 | 0.9634 | 0.9934 | 0 |
| raw mask + median5 + fill | 0.9111 | 0.8367 | 0.9971 | 0.8388 | 0 |
| Sobel gate + median3 + contour fill | 0.7131 | 0.5541 | 0.5541 | 0.9999 | 0 |

* **No CLOSE.** Its brush has to be small relative to the *object*, and a car is about 20 px in
  a 240 px frame — even a 15 px brush is most of a car. A closing wide enough to bridge a gap
  in one car also bridges the road between two cars, and F1 barely notices: in the table above
  precision falls 0.9863 → 0.9634 while recall rises 0.9823 → 0.9934, and F1 moves by 0.006.
  The defect is invisible in the summary and obvious in the picture.
* **No contour fill.** Filling every external contour solid is right for one person and wrong
  for traffic — it swallows the road between cars. Precision 0.55, recall 1.0.
* **Threshold on confidence, not on MOG2's own decision.** MOG2 calls a pixel background on
  *any* match, however rarely that colour was seen (`bg_prob > 0`). Requiring half the weight
  instead is worth +7 F1 and costs one comparison per pixel.

In [ ]:
if HAS_FRAMES:
    import numpy as np, cv2, matplotlib.pyplot as plt
    from gmm_mask import GMM_Mask_Numba
    from utils.post_processing import fill_holes, threshold_bg_prob

    N = 560
    frames, src = load_sequence(N)
    h, w = frames[0].shape[:2]
    m = GMM_Mask_Numba(h, w)
    for bgr in frames:
        mask, bg_prob, _ = m.apply(cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb))

    raw  = np.asarray(mask)
    thr  = threshold_bg_prob(np.asarray(bg_prob))
    med  = cv2.medianBlur(thr, 5)
    fill = fill_holes(med)

    stages = [("input", cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)),
              ("raw MOG2 decision", raw), ("threshold bg_prob<0.5", thr),
              ("median 5", med), ("+ fill_holes", fill)]
    if HAS_DATA:
        gt = cv2.imread(f"{HIGHWAY}/groundtruth/gt{len(frames):06d}.png", 0)
        if gt is not None:
            stages.append(("ground truth", gt))

    fig, ax = plt.subplots(1, len(stages), figsize=(2.5 * len(stages), 2.4))
    for a, (t, im) in zip(ax, stages):
        a.imshow(im, cmap=None if im.ndim == 3 else "gray")
        a.set_title(t, fontsize=8); a.axis("off")
    fig.suptitle(f"source: {src}", fontsize=8, y=1.04)
    plt.tight_layout(); plt.show()

    def holes(x):
        n, _ = cv2.connectedComponents((fill_holes(x) & ~x).astype(np.uint8))
        return n - 1
    h_raw, h_med, h_fill = holes(raw), holes(med), holes(fill)
    print(f"holes in the mask:  raw {h_raw:4d}   median5 {h_med:4d}   "
          f"after fill {h_fill:4d}")
    if not HAS_DATA and h_raw == 0:
        print()
        print("  Note: zero holes everywhere, which does NOT support the argument above.")
        print("  The synthetic fallback draws flat rectangles, and a flat object has no")
        print("  interior for MOG2 to miss. Hollowing is what happens to a real car roof:")
        print("  a large uniform surface where motion changes nothing, so the middle comes")
        print("  back empty. The 77.2 holes/frame figure quoted in the report is from the")
        print("  CDnet sequence; run this cell with HIGHWAY_DIR set to reproduce it.")
else:
    print("no frames available — upload a clip or the CDnet sequence")

## 6. Where the time actually goes

This is the result the project turns on, and it was a surprise.

Once v1 moved the mask stages onto the device, **the model kernel stopped being the
bottleneck and the host did not**. Profiling a 1080p frame put **81.8% of the time on the host**
and 18.2% on the GPU. Making the GPU infinitely fast from there buys 1.22×.

> *One-off measurement, kept because it is what motivated v1.* No profiler in this repo
> reproduces it; the per-stage split in §6.2 is the version you can run. Both say the same
> thing, which is why the older number is retained rather than relied on.

So the optimisation target was never the arithmetic. It was the bus and the host:

| version | what runs where |
|---|---|
| **v0** | mask on the GPU; threshold, median, fill, blur, composite on the host |
| **v1** | + threshold, median, blur and composite as CUDA kernels; colour conversion on the device |
| **v2** | + threshold **fused** into the model kernel's epilogue; median and blur shared-memory tiled |

In [ ]:
import bench_post as bp
v0_mb, gpu_mb = bp.bytes_per_frame(1080, 1920)
print(f"bus traffic per frame at 1080p:  v0 {v0_mb:.2f} MB   v1/v2 {gpu_mb:.2f} MB "
      f"({(1-gpu_mb/v0_mb):.1%} less, {v0_mb/gpu_mb:.2f}x)")
print()
print("  v0   planar float32 frame up   12 B/px      = %6.2f MB" % (1920*1080*12/1e6))
print("       mask down                  1 B/px      = %6.2f MB" % (1920*1080*1/1e6))
print("       bg_prob down               4 B/px      = %6.2f MB" % (1920*1080*4/1e6))
print("  v1/2 BGR frame up               3 B/px      = %6.2f MB" % (1920*1080*3/1e6))
print("       mask down                  1 B/px      = %6.2f MB" % (1920*1080*1/1e6))
print("       filled mask up             1 B/px      = %6.2f MB" % (1920*1080*1/1e6))
print("       composite down             3 B/px      = %6.2f MB" % (1920*1080*3/1e6))

import matplotlib.pyplot as plt, numpy as np
sizes = [(854,480),(1280,720),(1920,1080)]
lbl = ["480p","720p","1080p"]
a = [bp.bytes_per_frame(h,w)[0] for w,h in sizes]
b = [bp.bytes_per_frame(h,w)[1] for w,h in sizes]
x = np.arange(3); plt.figure(figsize=(5,2.6))
plt.bar(x-0.2, a, 0.4, label="v0", color=C["v0"])
plt.bar(x+0.2, b, 0.4, label="v1 / v2", color=C["v2"])
plt.xticks(x, lbl); plt.ylabel("MB across the bus / frame"); plt.legend()
plt.title("Bus traffic, computed from the array shapes")
plt.tight_layout(); plt.show()

### 6.1 Kernel 2 — the separable blur, and why it is integer arithmetic

The blur was the last host stage. Moving it required reproducing `cv2.GaussianBlur` exactly,
and that turned out to be the interesting part:

**`cv2.GaussianBlur` on a uint8 image is not a floating-point convolution.** It quantises the
kernel to Q8 and runs fixed point, and the quantiser is *cumulative*, not per-tap:

$$KQ[i] = \mathrm{round}(256\cdot\mathrm{cumsum}(k)[i]) - \mathrm{round}(256\cdot\mathrm{cumsum}(k)[i-1])$$

Rounding each tap on its own — the obvious implementation — is wrong at four of the fifteen
positions (two distinct coefficients, each appearing twice by symmetry).
And a *correct* float64 convolution still disagrees with OpenCV on about **16% of pixels, by one
grey level**: that 16% is OpenCV's own departure from an ideal Gaussian, and reproducing OpenCV
means reproducing it.

Because the whole path is integer, the equality holds **independently of GPU architecture and
compiler flags** — unlike the MOG2 kernel, whose float32 parity depends on how FMAs contract.
This is the strongest correctness claim in the project.

In [ ]:
import cv2, numpy as np
from gmm_mask.gpu.blur_kernels import gaussian_kernel_q8, blur_reference
from settings import BLUR_KSIZE, BLUR_SIGMA

k  = cv2.getGaussianKernel(BLUR_KSIZE, BLUR_SIGMA).ravel()
KQ = gaussian_kernel_q8()
per_tap = np.rint(k*256).astype(int)
print("cumulative (what OpenCV does):", KQ,        "sum", KQ.sum())
print("per-tap    (the wrong way)   :", per_tap, "sum", per_tap.sum())
print("differ at taps:", np.flatnonzero(KQ != per_tap))

rng = np.random.default_rng(0); bad = tot = 0
for s in range(6):
    img = rng.integers(0,256,(120,160,3),dtype=np.uint8)
    d = blur_reference(img, KQ) != cv2.GaussianBlur(img,(BLUR_KSIZE,BLUR_KSIZE),BLUR_SIGMA)
    bad += int(d.sum()); tot += d.size
print(f"\nour fixed-point chain vs cv2.GaussianBlur : {bad} / {tot} pixels differ")

img = rng.integers(0,256,(120,160,3),dtype=np.uint8)
ideal = np.floor(cv2.sepFilter2D(img.astype(np.float64), cv2.CV_64F, k, k,
                borderType=cv2.BORDER_REFLECT_101) + 0.5).astype(np.uint8)
d = np.abs(ideal.astype(int) - blur_reference(img,KQ).astype(int))
print(f"an *ideal* float Gaussian vs cv2          : {(d!=0).mean():.1%} of pixels differ, "
      f"max {d.max()} grey level")

In [ ]:
if HAS_GPU:
    !python -m pytest tests/test_blur.py -q 2>&1 | tail -6
else:
    print("no GPU — run tests/test_blur.py under NUMBA_ENABLE_CUDASIM=1 instead")

### 6.2 The measurement

Correctness first, on models nothing has timed: **every frame** of v1 and v2 compared against
the host chain, mask and composite. Only then is anything timed, and each timing repeat rebuilds
the model so that no repeat measures a more converged mixture than the one before it.

In [ ]:
if HAS_GPU:
    !python bench_post.py --sizes 480 720 1080 2>&1 | tail -60
else:
    print("skipped — needs a real GPU")

In [ ]:
if HAS_GPU:
    !python bench_post.py --sizes 480 --with-sequential --skip-equivalence 2>&1 | tail -12
else:
    print("skipped — needs a real GPU")

### The benchmark charts

The cell below draws the speedup and throughput charts from the **recorded T4
numbers** in `RESULTS-T4.md` — quoted, because charts need a GPU-measured
input and this notebook must also render on a CPU-only machine. On a GPU, the
two cells above just re-measured the same table live; compare their printout
against `T4_RESULTS` and re-run `bench_post.py` to regenerate the record.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

# Quoted from RESULTS-T4.md (Tesla T4, commit noted there). Keys: ms/frame, FPS.
T4_RESULTS = {
    "sizes": ["480p", "720p", "1080p"],
    "v0": {"ms": [10.22, 20.82, 55.03], "fps": [97.9, 48.0, 18.2]},
    "v1": {"ms": [3.74, 6.98, 13.75],   "fps": [267.3, 143.3, 72.8]},
    "v2": {"ms": [3.26, 6.02, 11.26],   "fps": [307.1, 166.2, 88.8]},
}

x = np.arange(3); w = 0.26
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for i, (k, col) in enumerate([("v0", "#95a5a6"), ("v1", C["v1"]), ("v2", C["v2"])]):
    b = a1.bar(x + (i - 1) * w, T4_RESULTS[k]["ms"], w, label=k, color=col)
    a1.bar_label(b, fmt="%.2f", fontsize=7, fontweight="bold")
    b = a2.bar(x + (i - 1) * w, T4_RESULTS[k]["fps"], w, label=k, color=col)
    a2.bar_label(b, fmt="%.0f", fontsize=7, fontweight="bold")
a1.set_xticks(x); a1.set_xticklabels(T4_RESULTS["sizes"])
a1.set_ylabel("ms / frame"); a1.set_title("Time per frame (lower is better)")
a2.set_xticks(x); a2.set_xticklabels(T4_RESULTS["sizes"])
a2.axhline(30, color="black", lw=1, ls=":")
a2.text(2.35, 34, "30 FPS target", ha="right", fontsize=8, style="italic")
a2.set_ylabel("frames / second"); a2.set_title("Throughput (higher is better)")
a1.legend(frameon=False); a2.legend(frameon=False)
plt.tight_layout(); plt.show()

# Recomputed from the table's rounded entries, so the last digit can differ
# from RESULTS-T4.md's ratios (which divide the unrounded measurements —
# e.g. 3.13x here vs 3.14x there at 480p). The record is authoritative.
sp = [a / b for a, b in zip(T4_RESULTS["v0"]["ms"], T4_RESULTS["v2"]["ms"])]
print("v2 over v0:", " · ".join(f"{s}: {r:.2f}x" for s, r in zip(T4_RESULTS["sizes"], sp)))

### 6.3 Predicted, then measured — including the predictions that were wrong

The predictions below were written down **before** the T4 run, which is the only way a
prediction is worth anything.

| | predicted | measured on a T4 |
|---|---|---|
| host `cv2` blur+composite @1080p | 4–8 ms | **13.7 ms** |
| GPU blur kernels @1080p | 0.2–0.4 ms | **1.55 ms** (tiled) |
| **tiled vs naive** | *"probably close to naive"* | **2.36×** |
| ingest+blur saved @1080p | 4–8 ms | **~34 ms** |
| bottleneck afterwards | host `fill_holes`, not the blur | **confirmed, 36.7%** |
| bus traffic @1080p | −38.5% | **−52.9%** |

**Five of the six missed** — only the bottleneck call held. Three underestimated the host and
the bus, one underestimated what a GPU kernel costs, and one went the other way entirely. That is the project's actual finding, and it is why the optimisation
target ended up being the bus and the host rather than the arithmetic.

**The tiling prediction was wrong cleanly.** The written reasoning was that L2 already serves
row-strided reuse well, so staging a shared tile would buy little. It buys **2.36×** — at three
resolutions, under two different measurement protocols (batched and interleaved). The timing is measured; the cache-level explanation is our reading of it, not a
profiler result.

**The ingest win is mostly not the kernel**, and saying so matters. Decomposed at 1080p:

| cause of the 21.6 ms saved | ms | share |
|---|---|---|
| host colour convert + float32 transpose removed | 16.51 | 76% |
| 12 B/px → 3 B/px upload | 3.68 | 17% |
| device allocation removed (v0 allocates 25 MB every frame) | 1.56 | 7% |
| conversion kernel added back | −0.18 | — |

The conversion kernel itself costs **0.184 ms**. Being nearly free is exactly what made the
3 B/px upload possible, because the conversion had to go *somewhere*.

**And one speedup shrank when the baseline got fairer.** An earlier run put v2 at 5.72× over v0
at 1080p; it is 4.89× now, and nothing about v2 got slower. v0 got faster, because the host path
was converting to float32 twice — 2.2 ms a frame for an identical array. The old number was
partly measuring waste in the thing being beaten.

Full numbers, reproduction commands and the on-hardware correctness sweep are in
`RESULTS-T4.md`.

## 7. The stage that stays on the CPU, and why that is the point

`fill_holes` floods the background inward from the image border and ORs back the complement:
whatever the flood cannot reach is a hole. It fills a hole of *any* size without moving the
silhouette, and — unlike a morphological CLOSE — it cannot bridge two separate objects.

OpenCV implements it as a scan-line flood fill, which is sequential. A data-parallel formulation
does exist — morphological reconstruction by dilation — so the question is not whether it *can*
be parallelised but whether doing so is worth it. It needs **one dilate per pixel of propagation
distance**, and each dilate is a grid-wide step:

`bench_fill.py` implements both, checks that they agree pixel-for-pixel — a reconstruction that
gives a different answer is not a slower alternative, it is a wrong one — and times them. The
cell below runs it.

The number that matters is not the ratio but the **pass count**: each dilate is a full-frame,
grid-wide step, and no amount of GPU shortens the *sequence* of them.

Knowing which stage *not* to move is as much a result as any kernel here. The same discipline
retired GrabCut: implemented, measured at +2.1 F1 for roughly 140× the cost and 5 frames of
outright empty mask, and cut. *(That one is quoted from an earlier experiment — the GrabCut
implementation has since been removed from this branch, so unlike the flood fill it is not
reproducible from this checkout.)*

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "bench_fill.py", "--sizes", "240", "480", "720", "1080"],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print("FAILED:", r.stderr[-1500:])

## 8. The result

Composite output: sharp vehicles, blurred road.

In [ ]:
if HAS_FRAMES:
    import numpy as np, cv2, matplotlib.pyplot as plt
    from gmm_mask import GMM_Mask_Numba
    from utils.post_processing import refine_mask, background_blur
    from settings import BLUR_KSIZE, BLUR_SIGMA

    frames, src = load_sequence(600)
    h, w = frames[0].shape[:2]
    m = GMM_Mask_Numba(h, w)
    for bgr in frames:
        mask, bg_prob, _ = m.apply(cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb))

    refined = refine_mask(np.asarray(mask), bg_prob=np.asarray(bg_prob))
    out = background_blur(bgr, refined, BLUR_KSIZE, BLUR_SIGMA)

    # The model was fed YCrCb, so its learned means are YCrCb. Converting them
    # as if they were BGR renders the road blue.
    bg = cv2.cvtColor(m.background_image(), cv2.COLOR_YCrCb2RGB)

    fig, ax = plt.subplots(1, 4, figsize=(13, 2.6))
    for a, (t, im) in zip(ax, [("original", cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)),
                               ("mask", refined),
                               ("learned background", bg),
                               ("composite", cv2.cvtColor(out, cv2.COLOR_BGR2RGB))]):
        a.imshow(im, cmap=None if im.ndim == 3 else "gray")
        a.set_title(t); a.axis("off")
    fig.suptitle(f"source: {src}", fontsize=8, y=1.04)
    plt.tight_layout(); plt.show()
else:
    print("no frames available — upload a clip or the CDnet sequence")

## 9. What we learned

1. **The bottleneck was the host, not the arithmetic.** After v1 the host held 81.8% of the
   1080p frame budget. Every subsequent decision followed from that measurement rather than from
   intuition about which kernel looked expensive.

2. **Reproducing a library exactly is harder, and more informative, than beating it.**
   `cv2.GaussianBlur` on uint8 is fixed-point with a cumulative quantiser; `cv2.cvtColor` is
   fixed-point too. Both are reproducible bit-for-bit — and finding that out is what let the
   colour conversion move to the device, which is what made the 3 B/px upload possible.

3. **Some stages should not be parallelised, and measuring that is the contribution.**
   `fill_holes` costs 593 dependent full-frame passes as a data-parallel reconstruction — 112× the wall-clock on the T4 host, 287× on an Apple host, with the pass count identical on both. GrabCut bought +2.1 F1 for
   141× the cost. Both are reported as measurements, not omissions.

4. **A metric can hide a defect.** A 15×15 closing merged two vehicles into one blob while F1
   stayed near 0.96. Every candidate was inspected, not just scored.

5. **We wrote our predictions down, and five of the six missed.** We predicted shared-memory
   tiling would barely beat the naive blur, because L2 should already serve the row reuse.
   It wins 2.36×, essentially flat across the three resolutions (2.36 / 2.36 / 2.37). Recording the prediction is what turned
   that into a finding instead of a number nobody questioned.

### Still open

CUDA streams overlapping the next frame's upload with this frame's host flood fill — the D2H →
`fill_holes` → H2D round trip is a hard sync point in the middle of every frame, and it is now
the largest remaining item. Adaptive K per pixel (Zivkovic's complexity reduction) is
implemented but unscored on this dataset.